In [1]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
PROJECT_ID = "project-5eb4889f-74fb-4ad5-a76"
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-5eb4889f-74fb-4ad5-a76


In [2]:
raw = client.query(f"""
    SELECT ingested_at, business_date, payload
    FROM `{PROJECT_ID}.nbu_raw.raw_rates`
""").to_dataframe()

print(raw.shape)

parsed = pd.json_normalize(raw["payload"].map(json.loads))
parsed["ingested_at"] = raw["ingested_at"]

print(parsed.shape)
parsed.head()

(270, 3)
(270, 7)


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cc,exchangedate,r030,rate,special,txt,ingested_at
0,DZD,25.09.2026,12,0.33625,NaN,Алжирський динар,2026-09-24 14:11:15.376348+00:00
1,AUD,25.09.2026,36,31.59710,NaN,Австралійський долар,2026-09-24 14:11:15.376348+00:00
2,BDT,25.09.2026,50,0.36557,NaN,Така,2026-09-24 14:11:15.376348+00:00
3,CAD,25.09.2026,124,31.85520,NaN,Канадський долар,2026-09-24 14:11:15.376348+00:00
4,CNY,25.09.2026,156,6.70050,NaN,Юань Женьміньбі,2026-09-24 14:11:15.376348+00:00


In [3]:
df = parsed.copy()
df["currency_code"] = df["cc"].str.strip().str.upper()
df["currency_name"] = df["txt"].str.strip()
df["r030"] = df["r030"].astype("Int64")

df[["currency_code", "currency_name", "r030", "ingested_at"]].head()

,currency_code,currency_name,r030,ingested_at
0,DZD,Алжирський динар,12,2026-09-24 14:11:15.376348+00:00
1,AUD,Австралійський долар,36,2026-09-24 14:11:15.376348+00:00
2,BDT,Така,50,2026-09-24 14:11:15.376348+00:00
3,CAD,Канадський долар,124,2026-09-24 14:11:15.376348+00:00
4,CNY,Юань Женьміньбі,156,2026-09-24 14:11:15.376348+00:00


In [4]:
dim = (df.sort_values("ingested_at")
          .drop_duplicates(subset="currency_code", keep="last"))

print(dim.shape)
dim[["currency_code", "currency_name", "r030"]].head()

(45, 9)


,currency_code,currency_name,r030
155,NOK,Норвезька крона,578
154,NZD,Новозеландський долар,554
153,MDL,Молдовський лей,498
152,MXN,Мексиканське песо,484
151,MYR,Малайзійський ринггіт,458


In [5]:
dim = dim.sort_values("currency_code").reset_index(drop=True)
dim["currency_key"] = dim.index + 1
dim = dim[["currency_key", "currency_code", "currency_name", "r030"]]

dim.head()

,currency_key,currency_code,currency_name,r030
0,1,AED,Дирхам ОАЕ,784
1,2,AUD,Австралійський долар,36
2,3,AZN,Азербайджанський манат,944
3,4,BDT,Така,50
4,5,CAD,Канадський долар,124


In [6]:
unknown_row = pd.DataFrame([{
    "currency_key": -1,
    "currency_code": "N/A",
    "currency_name": "Unknown",
    "r030": -1,
}])

dim = pd.concat([dim, unknown_row], ignore_index=True)
dim.tail()

,currency_key,currency_code,currency_name,r030
41,42,XDR,СПЗ (спеціальні права запозичення),960
42,43,XPD,Паладій,964
43,44,XPT,Платина,962
44,45,ZAR,Ренд,710
45,-1,N/A,Unknown,-1


In [7]:
dim_schema = [
    bigquery.SchemaField("currency_key",  "INTEGER"),
    bigquery.SchemaField("currency_code", "STRING"),
    bigquery.SchemaField("currency_name", "STRING"),
    bigquery.SchemaField("r030",          "INTEGER"),
]

DIM_TABLE_ID = f"{PROJECT_ID}.nbu_dwh.dim_currency"

cfg = bigquery.LoadJobConfig(schema=dim_schema, write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(dim, DIM_TABLE_ID, job_config=cfg).result()

print("записано рядків:", len(dim))

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


записано рядків: 46


In [8]:
check_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT currency_code) AS uniq_codes,
  COUNTIF(currency_key = -1) AS has_unknown
FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
"""

client.query(check_sql).to_dataframe()

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,uniq_codes,has_unknown
0,46,46,1
